# Clasificación de pastores alemanes y otros perros

Este notebook prepara el dataset de imágenes para el problema binario de detección de pastores alemanes.

La idea es que todo quede parametrizado para que mañana puedas cambiar la raza objetivo sin rehacer el flujo completo.

## Decisión sobre el código preliminar

El bloque que venía en este notebook estaba escrito dentro de una celda markdown, así que no era reutilizable tal cual.

En lugar de copiarlo sin más, conviene transformarlo en un flujo parametrizado con estas etapas:

1. Descarga de al menos 1500 imágenes por clase.
2. Normalización a 224x224.
3. Eliminación de duplicados con `imagededup`.
4. Organización por carpetas para facilitar el cambio de clase en el futuro.

La configuración de clases, palabras clave y rutas queda centralizada en una sola sección.

In [1]:
from pathlib import Path
from typing import Dict, Tuple

PROJECT_ROOT = Path('.').resolve()
DATA_ROOT = PROJECT_ROOT / 'data' / 'dog_classification'
RAW_ROOT = DATA_ROOT / 'raw'
PROCESSED_ROOT = DATA_ROOT / 'processed'
TARGET_SIZE: Tuple[int, int] = (224, 224)
MIN_IMAGES_PER_CLASS = 1500

CLASS_SPECS: Dict[str, Dict[str, str]] = {
    'german_shepherd': {
        'keyword': 'German Shepherd dog',
        'raw_dir': str(RAW_ROOT / 'german_shepherd'),
        'processed_dir': str(PROCESSED_ROOT / 'german_shepherd'),
    },
    'other_dogs': {
        'keyword': 'dog breeds',
        'raw_dir': str(RAW_ROOT / 'other_dogs'),
        'processed_dir': str(PROCESSED_ROOT / 'other_dogs'),
    },
}

for spec in CLASS_SPECS.values():
    Path(spec['raw_dir']).mkdir(parents=True, exist_ok=True)
    Path(spec['processed_dir']).mkdir(parents=True, exist_ok=True)

for folder in (DATA_ROOT, RAW_ROOT, PROCESSED_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

print('Ruta base del proyecto:', DATA_ROOT)
print('Tamaño objetivo:', TARGET_SIZE)
print('Clases configuradas:', ', '.join(CLASS_SPECS))

Ruta base del proyecto: /home/martin/Documents/GitHub/AI-Frameworks/data/dog_classification
Tamaño objetivo: (224, 224)
Clases configuradas: german_shepherd, other_dogs


In [2]:
# Si faltan dependencias, instálalas antes de ejecutar estas celdas:
# %pip install icrawler pillow imagededup tqdm

from pathlib import Path
from typing import Iterable

from icrawler.builtin import BingImageCrawler, GoogleImageCrawler
from PIL import Image, ImageOps
from imagededup.methods import PHash
from tqdm.auto import tqdm

SEARCH_ENGINES = {
    'google': GoogleImageCrawler,
    'bing': BingImageCrawler,
}


def crawl_images(keyword: str, output_dir: Path, max_num: int, engine: str = 'bing') -> None:
    """Descarga imágenes para una clase concreta."""
    crawler_cls = SEARCH_ENGINES[engine]
    crawler = crawler_cls(storage={'root_dir': str(output_dir)})
    crawler.crawl(keyword=keyword, max_num=max_num)


def resize_image(image_path: Path, target_size: tuple[int, int]) -> None:
    """Convierte a RGB y ajusta el tamaño sin deformar la imagen."""
    with Image.open(image_path) as image:
        cleaned = ImageOps.fit(image.convert('RGB'), target_size, method=Image.Resampling.LANCZOS)
        cleaned.save(image_path, quality=95)


def resize_folder(folder: Path, target_size: tuple[int, int]) -> None:
    for image_path in tqdm(list(folder.rglob('*'))):
        if image_path.is_file() and image_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
            resize_image(image_path, target_size)


def remove_duplicates(folder: Path) -> None:
    """Elimina duplicados dentro de una carpeta usando imagededup."""
    deduper = PHash()
    duplicate_map = deduper.find_duplicates(image_dir=str(folder), scores=False)
    duplicate_names = {duplicate_name for duplicates in duplicate_map.values() for duplicate_name in duplicates}

    for duplicate_name in duplicate_names:
        candidate = folder / duplicate_name
        if candidate.exists():
            candidate.unlink()


def count_images(folder: Path) -> int:
    return sum(
        1
        for image_path in folder.rglob('*')
        if image_path.is_file() and image_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}
    )


def build_dataset(class_specs: dict[str, dict[str, str]], minimum_images: int, target_size: tuple[int, int], engine: str = 'bing') -> None:
    for class_name, spec in class_specs.items():
        raw_dir = Path(spec['raw_dir'])
        processed_dir = Path(spec['processed_dir'])
        keyword = spec['keyword']

        raw_dir.mkdir(parents=True, exist_ok=True)
        processed_dir.mkdir(parents=True, exist_ok=True)

        print(f'[{class_name}] descargando {minimum_images} imágenes con el texto: {keyword}')
        crawl_images(keyword=keyword, output_dir=raw_dir, max_num=minimum_images, engine=engine)

        print(f'[{class_name}] ajustando tamaño a {target_size}')
        resize_folder(raw_dir, target_size)

        print(f'[{class_name}] eliminando duplicados')
        remove_duplicates(raw_dir)

        final_count = count_images(raw_dir)
        print(f'[{class_name}] imágenes finales en bruto: {final_count}')
        if final_count == 0:
            raise RuntimeError(f'[{class_name}] no se descargaron imágenes. Revisa la conexión o el motor "{engine}".')


# Ejecución real del notebook.
build_dataset(CLASS_SPECS, MIN_IMAGES_PER_CLASS, TARGET_SIZE, engine='bing')


/home/martin/Documents/GitHub/AI-Frameworks/.venv-pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-02 12:40:01,723 - WARNING - icrawler.crawler - Due to Bing's limitation, you can only get the first 1000 result. "max_num" has been automatically set to 1000
2026-07-02 12:40:01,727 - INFO - icrawler.crawler - start crawling...
2026-07-02 12:40:01,728 - INFO - icrawler.crawler - starting 1 feeder threads...
2026-07-02 12:40:01,730 - INFO - icrawler.crawler - starting 1 parser threads...
2026-07-02 12:40:01,732 - INFO - icrawler.crawler - starting 1 downloader threads...


[german_shepherd] descargando 1500 imágenes con el texto: German Shepherd dog


2026-07-02 12:40:02,290 - INFO - parser - parsing result page https://www.bing.com/images/async?q=German Shepherd dog&first=0
2026-07-02 12:40:02,339 - INFO - downloader - skip downloading file 000001.jpg
2026-07-02 12:40:02,343 - INFO - downloader - skip downloading file 000002.jpg
2026-07-02 12:40:02,348 - INFO - downloader - skip downloading file 000003.jpg
2026-07-02 12:40:02,354 - INFO - downloader - skip downloading file 000004.jpg
2026-07-02 12:40:02,358 - INFO - downloader - skip downloading file 000005.jpg
2026-07-02 12:40:02,361 - INFO - downloader - skip downloading file 000006.jpg
2026-07-02 12:40:02,363 - INFO - downloader - skip downloading file 000007.jpg
2026-07-02 12:40:02,370 - INFO - downloader - skip downloading file 000008.jpg
2026-07-02 12:40:02,375 - INFO - downloader - skip downloading file 000009.jpg
2026-07-02 12:40:02,377 - INFO - downloader - skip downloading file 000010.jpg
2026-07-02 12:40:02,381 - INFO - downloader - skip downloading file 000011.jpg
2026-

[german_shepherd] ajustando tamaño a (224, 224)


100%|██████████| 899/899 [00:10<00:00, 83.55it/s] 
2026-07-02 12:45:35,800: INFO Start: Calculating hashes...
2026-07-02 12:45:35,800 - INFO - imagededup.methods.hashing - Start: Calculating hashes...


...

[other_dogs] ajustando tamaño a (224, 224)


100%|██████████| 603/603 [00:04<00:00, 129.09it/s]
2026-07-02 12:48:39,880: INFO Start: Calculating hashes...
2026-07-02 12:48:39,880 - INFO - imagededup.methods.hashing - Start: Calculating hashes...


[other_dogs] eliminando duplicados


100%|██████████| 603/603 [00:00<00:00, 1465.17it/s]
2026-07-02 12:48:40,487: INFO End: Calculating hashes!
2026-07-02 12:48:40,487 - INFO - imagededup.methods.hashing - End: Calculating hashes!
2026-07-02 12:48:40,490: INFO Start: Evaluating hamming distances for getting duplicates
2026-07-02 12:48:40,490 - INFO - imagededup.methods.hashing - Start: Evaluating hamming distances for getting duplicates
2026-07-02 12:48:40,493: INFO Start: Retrieving duplicates using Cython Brute force algorithm
2026-07-02 12:48:40,493 - INFO - imagededup.handlers.search.retrieval - Start: Retrieving duplicates using Cython Brute force algorithm
100%|██████████| 603/603 [00:00<00:00, 14031.74it/s]
2026-07-02 12:48:40,644: INFO End: Retrieving duplicates using Cython Brute force algorithm
2026-07-02 12:48:40,644 - INFO - imagededup.handlers.search.retrieval - End: Retrieving duplicates using Cython Brute force algorithm
2026-07-02 12:48:40,647: INFO End: Evaluating hamming distances for getting duplicates


[other_dogs] imágenes finales en bruto: 603
